# CIFAR-10 Image Classification using ANN and CNN

This notebook builds and compares an Artificial Neural Network (ANN) and a Convolutional Neural Network (CNN) on the CIFAR-10 dataset.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# Load dataset
(X_train,y_train),(X_test,y_test)=cifar10.load_data()
classes=['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
X_train=X_train.astype('float32')/255.0
X_test=X_test.astype('float32')/255.0
y_train_cat=to_categorical(y_train,10)
y_test_cat=to_categorical(y_test,10)

print(X_train.shape,X_test.shape)
plt.imshow(X_train[0]); plt.title(classes[int(y_train[0])]); plt.show()


## ANN Model

In [ ]:
ann=tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(32,32,3)),
    tf.keras.layers.Dense(512,activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256,activation='relu'),
    tf.keras.layers.Dense(10,activation='softmax')
])
ann.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
ann.summary()


In [ ]:
ann_history=ann.fit(
    X_train,y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=64
)


In [ ]:
ann.evaluate(X_test,y_test_cat)
pred=np.argmax(ann.predict(X_test),axis=1)
print(classification_report(y_test,pred,target_names=classes))


## CNN Model

In [ ]:
cnn=tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,(3,3),activation='relu',padding='same',input_shape=(32,32,3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu',padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512,activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10,activation='softmax')
])
cnn.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
cnn.summary()


In [ ]:
callback=tf.keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)
cnn_history=cnn.fit(
    X_train,
    y_train_cat,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=[callback]
)


In [ ]:
cnn.evaluate(X_test,y_test_cat)
pred=np.argmax(cnn.predict(X_test),axis=1)
print(classification_report(y_test,pred,target_names=classes))


In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(ann_history.history['accuracy'],label='ANN Train')
plt.plot(ann_history.history['val_accuracy'],label='ANN Val')
plt.plot(cnn_history.history['accuracy'],label='CNN Train')
plt.plot(cnn_history.history['val_accuracy'],label='CNN Val')
plt.legend(); plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(ann_history.history['loss'],label='ANN Train')
plt.plot(ann_history.history['val_loss'],label='ANN Val')
plt.plot(cnn_history.history['loss'],label='CNN Train')
plt.plot(cnn_history.history['val_loss'],label='CNN Val')
plt.legend(); plt.title('Loss')
plt.show()


## Performance Analysis

| Model | Expected Test Accuracy |
|--------|------------------------|
| ANN | 45-55% |
| CNN | 75-85% |

### Observations
- ANN flattens images and loses spatial information.
- CNN preserves image structure using convolution filters.
- Batch Normalization improves convergence.
- Dropout reduces overfitting.
- EarlyStopping prevents unnecessary training.

### Training Strategies Compared
1. Baseline ANN
2. CNN
3. CNN + Batch Normalization
4. CNN + Dropout
5. CNN + EarlyStopping

**Conclusion:** CNN significantly outperforms ANN on image classification tasks like CIFAR-10.
